# DS2002 · Cleaning Clinic

**Studio — 2026-09-23 · Fall 2026**  
**Class time:** 45 minutes

---

## Write the pipeline, then defend it

Monday I made the cleaning decisions and told you what they were. Today you make them, and the output is two things: a clean frame, and a **decision log** that says what you did to whose rows and why.

The log is not paperwork. On the midterm your team will disagree about whether a refund counts, and the log is what turns that into a two-minute conversation instead of an afternoon of re-deriving numbers.

Every step below follows the same three-part shape: **do it, count what you changed, log the decision.**

In [3]:
import pandas as pd, numpy as np
from io import StringIO
raw = '''order_id,item,category,qty,price,ts
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
2,cheese burger,food,1,7.5,09/05/2026 12:40
3,Foam Finger,Merch,NULL,12,2026-09-05 13:00:00
4,UVA T-Shirt ,Apparel,2,$24.00,2026-09-05 13:05
5,Rain Poncho,RainGear,-3,6,2026-09-05T13:20:00
6,rain poncho,rain-gear,4,$6.00,
7,,Merch,1,12,2026-09-05T14:00:00'''
df = pd.read_csv(StringIO(raw))
df

,order_id,item,category,qty,price,ts
0,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
1,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
2,2,cheese burger,food,1.0,7.5,09/05/2026 12:40
3,3,Foam Finger,Merch,NaN,12,2026-09-05 13:00:00
4,4,UVA T-Shirt,Apparel,2.0,$24.00,2026-09-05 13:05
5,5,Rain Poncho,RainGear,-3.0,6,2026-09-05T13:20:00
6,6,rain poncho,rain-gear,4.0,$6.00,NaN
7,7,NaN,Merch,1.0,12,2026-09-05T14:00:00


### Set up the log

Run this first. Each step calls `log()` with what happened and how many rows it touched.

In [4]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

def show_log():
    return pd.DataFrame(DECISIONS)

raw_rows = len(df)
print('starting with', raw_rows, 'rows')

starting with 8 rows


### Step 0 — take inventory

**TODO:** print the shape, the dtypes, the null count per column, and the number of exact duplicate rows. Do not skip this — the rest of the studio depends on knowing what you have.

In [5]:
# Print DataFrame shape
print(f"Shape: {df.shape}")
print("-" * 30)

# Print data types
print("Data Types:")
print(df.dtypes)
print("-" * 30)

# Print null count per column
print("Null Count per Column:")
print(df.isnull().sum())
print("-" * 30)

# Print exact duplicate rows count
print(f"Exact Duplicate Rows: {df.duplicated().sum()}")

Shape: (8, 6)
------------------------------
Data Types:
order_id      int64
item            str
category        str
qty         float64
price           str
ts              str
dtype: object
------------------------------
Null Count per Column:
order_id    0
item        1
category    0
qty         1
price       0
ts          1
dtype: int64
------------------------------
Exact Duplicate Rows: 1


**What is wrong with this data?** List at least five specific problems:

1. _..._
2. _..._
3. _..._
4. _..._
5. _..._

### Step 1 — duplicates

**TODO:** drop exact duplicate rows into a new frame called `clean`, then log how many you removed. Use `.copy()` so later assignments do not warn.

In [6]:
# Drop exact duplicate rows into `clean` using .copy()
clean = df.drop_duplicates().copy()

# Calculate how many exact duplicate rows were removed
removed = len(df) - len(clean)

# Log the decision
log('duplicates', 'dropped exact duplicate rows', removed)

[duplicates] dropped exact duplicate rows (1 row(s))


### Step 2 — price into a real number

**TODO:** strip the dollar signs and any stray whitespace, then convert to float. Assert the dtype afterward so you find out now if a stray character survived.

In [7]:
clean['price'] = clean['price'].astype(str).str.replace('$', '', regex=False).str.strip().astype(float)

assert clean['price'].dtype == float

# Log the transformation
log('price_cleaning', 'stripped dollar signs/whitespace and cast to float', len(clean))

[price_cleaning] stripped dollar signs/whitespace and cast to float (7 row(s))


### Step 3 — quantity, and two decisions

**TODO:** coerce `qty` to numeric. Then decide, separately:

- what to do with the row that has no quantity
- what to do with the refund (negative quantity)

Log each decision with its row count. There is no single right answer — there is only an answer you can defend.

In [8]:
# Coerce qty to numeric
clean['qty'] = pd.to_numeric(clean['qty'], errors='coerce')

# Count issues before handling
missing = clean['qty'].isna().sum()
negative = (clean['qty'] < 0).sum()

# Decision 1: Drop rows with missing quantity (cannot calculate revenue without quantity)
clean = clean.dropna(subset=['qty']).copy()
log('missing_qty', 'dropped rows with missing/null quantity', missing)

# Decision 2: Keep refunds (do nothing)
log('negative_qty', 'keep refunds (do nothing)', negative)

[missing_qty] dropped rows with missing/null quantity (1 row(s))
[negative_qty] keep refunds (do nothing) (1 row(s))


### Step 4 — categories that mean one thing

**TODO:** normalize case and punctuation, then map the remaining variants with an explicit dict. Print the unique values before and after so the collapse is visible. Log how many distinct categories you started and ended with.

In [9]:
print('before:', sorted(clean['category'].dropna().unique()))

# Normalize case, strip spaces, and remove hyphens/punctuation
clean['category'] = clean['category'].astype(str).str.lower().str.strip().str.replace('-', '', regex=False)

# Map variations to canonical categories
CATEGORY_MAP = {
    'food': 'Food',
    'merch': 'Merch',
    'apparel': 'Apparel',
    'raingear': 'Rain Gear'
}
clean['category'] = clean['category'].map(CATEGORY_MAP)

print('after: ', sorted(clean['category'].unique()))

log('category_standardization', 'lowercased, stripped punctuation, and mapped to canonical categories', len(clean))

before: ['Apparel', 'Food', 'Merch', 'RainGear', 'food', 'rain-gear']
after:  ['Apparel', 'Food', 'Merch', 'Rain Gear']
[category_standardization] lowercased, stripped punctuation, and mapped to canonical categories (6 row(s))


### Step 5 — item names

**TODO:** same treatment for `item`. One product is spelled two ways, and one row has no item at all — decide what to do with it.

In [10]:
# Normalize whitespace and case
clean['item'] = clean['item'].astype(str).str.strip().str.lower()

# Standardize item name variants
ITEM_MAP = {
    'cheeseburger': 'Cheeseburger',
    'cheese burger': 'Cheeseburger',
    'foam finger': 'Foam Finger',
    'uva t-shirt': 'UVA T-Shirt',
    'rain poncho': 'Rain Poncho',
    'nan': np.nan
}
clean['item'] = clean['item'].map(ITEM_MAP)

# Drop rows missing an item name (cannot identify sold inventory)
missing_items = clean['item'].isna().sum()
clean = clean.dropna(subset=['item']).copy()

log('item_cleaning', 'standardized item name variations and dropped missing items', missing_items)

[item_cleaning] standardized item name variations and dropped missing items (1 row(s))


### Step 6 — timestamps

**TODO:** parse `ts` into real datetimes, coercing failures to `NaT`. Report how many failed. Then add an `hour` column, which is only possible once the column is a real datetime.

In [11]:
# Parse ts into datetimes, coercing unparseable dates to NaT
clean['ts'] = pd.to_datetime(clean['ts'], errors='coerce', format='mixed')

# Count parsing failures/missing timestamps
failed_ts = clean['ts'].isna().sum()

# Extract hour (will be NaN for NaT timestamps)
clean['hour'] = clean['ts'].dt.hour

log('timestamp_parsing', 'parsed datetime strings and extracted transaction hour', len(clean))
print(f'Failed/missing timestamps count: {failed_ts}')

[timestamp_parsing] parsed datetime strings and extracted transaction hour (5 row(s))
Failed/missing timestamps count: 1


### Step 7 — prove it

**TODO:** write at least five assertions that would catch a regression in this pipeline. Then compute `revenue` and print the totals.

In [19]:
# TODO: assertions

# TODO: clean['revenue'] = ...
# TODO: print rows, units, revenue, distinct categories
clean = clean.reset_index(drop=True)
clean

,order_id,item,category,qty,price,ts,hour
0,1,Cheeseburger,Food,2.0,7.5,2026-09-05 12:03:00,12.0
1,2,Cheeseburger,Food,1.0,7.5,2026-09-05 12:40:00,12.0
2,4,UVA T-Shirt,Apparel,2.0,24.0,2026-09-05 13:05:00,13.0
3,5,Rain Poncho,Rain Gear,-3.0,6.0,2026-09-05 13:20:00,13.0
4,6,Rain Poncho,Rain Gear,4.0,6.0,NaT,NaN


### Step 8 — the decision log

**TODO:** print your log. Then answer, in the markdown cell below: which single decision moved your revenue total the most, and what is the number both ways?

In [13]:
show_log()

,step,decision,rows
0,duplicates,dropped exact duplicate rows,1
1,price_cleaning,stripped dollar signs/whitespace and cast to f...,7
2,missing_qty,dropped rows with missing/null quantity,1
3,negative_qty,keep refunds (do nothing),1
4,category_standardization,"lowercased, stripped punctuation, and mapped t...",6
5,item_cleaning,standardized item name variations and dropped ...,1
6,timestamp_parsing,parsed datetime strings and extracted transact...,5


**The decision that mattered most:** _..._

**Revenue with it:** _..._  **Revenue without it:** _..._

---

## Checkpoint (participation)

Report your row count and revenue after cleaning, and the one decision that moved the total most.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [22]:
# Checkpoint
rows_after = 5            # TODO
revenue_after = (clean['qty'] * clean['price']).sum()         # TODO
biggest_decision = 'treating negative quantity (row 5) as a net refund instead of dropping or converting to positive'    # TODO: which choice moved the number most
revenue_other_way = 94.5   # TODO: the total if you had chosen differently

print('rows after cleaning:', rows_after)
print('revenue:', revenue_after)
print('decision that mattered:', biggest_decision)
print('revenue the other way:', revenue_other_way)

rows after cleaning: 5
revenue: 76.5
decision that mattered: treating negative quantity (row 5) as a net refund instead of dropping or converting to positive
revenue the other way: 94.5
